In [ ]:
# Check if GPU loaded
!nvidia-smi
import torch, sys
print(sys.version)
print(torch.__version__, torch.version.cuda, torch.cuda.is_available())

In [ ]:
# Construct python environment
%%bash

apt-get update
apt-get install -y python3.10 python3.10-venv python3.10-dev

python3.10 -m venv /content/py310

source /content/py310/bin/activate

python --version

pip install --upgrade pip setuptools wheel ninja cmake

pip install torch==2.4.0 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124

pip install gsplat==1.5.3 \
  -f https://docs.gsplat.studio/whl/pt24cu124

In [ ]:
# check if python environment setup successful
%%bash
source /content/py310/bin/activate

python -c "
import torch, gsplat
print('torch:', torch.__version__)
print('cuda:', torch.version.cuda)
print('cuda available:', torch.cuda.is_available())
print('gsplat:', gsplat.__file__)
"

In [ ]:
# gsplat installation into python environment
%%bash
source /content/py310/bin/activate

cd /content
rm -rf gsplat

git clone https://github.com/nerfstudio-project/gsplat.git
cd gsplat
git checkout v1.5.3

cd examples
pip install -r requirements.txt --no-build-isolation

In [ ]:
# Fetch images from Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Modify as necessary
%mkdir -p /content/nvs
!unzip -o /content/drive/MyDrive/NVS/colmap_no_mask_scene.zip -d /content/nvs

In [ ]:
# Check if unzip successful
!find /content/nvs/colmap_no_mask -maxdepth 3 -type f | head -50

In [ ]:
# Downsample images
%%bash
mkdir -p /content/nvs/colmap_no_mask/images_4
python - <<'PY'
from PIL import Image
from pathlib import Path

src = Path("/content/nvs/colmap_no_mask/images")
dst = Path("/content/nvs/colmap_no_mask/images_4")
dst.mkdir(exist_ok=True)

for p in src.iterdir():
    if p.suffix.lower() in [".jpg", ".jpeg", ".png"]:
        im = Image.open(p)
        im = im.resize((im.width // 4, im.height // 4))
        im.save(dst / p.name)
PY

In [ ]:
# %%bash
# cd /content/gsplat/examples
# mkdir -p ./results/no_mask
# cp -r /content/drive/MyDrive/NVS/no_mask/* /content/gsplat/examples/results/no_mask

In [ ]:
# Initial Training (Retrieve checkpoint at step 6999 and 29999)
%%bash
source /content/py310/bin/activate
cd /content/gsplat/examples

export MPLBACKEND=Agg
export PYTHONUNBUFFERED=1

mkdir -p ./results/no_mask

CUDA_VISIBLE_DEVICES=0 python -u simple_trainer.py default \
  --data_dir /content/nvs/colmap_no_mask \
  --data_factor 4 \
  --result_dir ./results/no_mask \
  --disable_viewer True

In [ ]:
# Modify simple_trainer to produce RGB-only trajectory frames
%%bash
cd /content/gsplat/examples

cp simple_trainer.py simple_trainer_modified.py

python - <<'PY'
from pathlib import Path

p = Path("simple_trainer_modified.py")
s = p.read_text()

# 1) Insert traj_frame_dir after writer creation
anchor1 = '        writer = imageio.get_writer(f"{video_dir}/traj_{step}.mp4", fps=30)\n'
insert1 = '''        writer = imageio.get_writer(f"{video_dir}/traj_{step}.mp4", fps=30)

        # save individual RGB trajectory frames
        traj_frame_dir = f"{cfg.result_dir}/traj_frames/step_{step}"
        os.makedirs(traj_frame_dir, exist_ok=True)
'''
if anchor1 not in s:
    raise SystemExit("Could not find writer creation line.")
s = s.replace(anchor1, insert1, 1)

# 2) Insert RGB-frame saving immediately after colors line
anchor2 = '            colors = torch.clamp(renders[..., 0:3], 0.0, 1.0)  # [1, H, W, 3]\n'
insert2 = '''            colors = torch.clamp(renders[..., 0:3], 0.0, 1.0)  # [1, H, W, 3]

            # save RGB-only frame for report figures
            rgb_frame = colors.squeeze(0).cpu().numpy()
            rgb_frame = (rgb_frame * 255).astype(np.uint8)
            imageio.imwrite(f"{traj_frame_dir}/traj_{step}_{i:04d}.png", rgb_frame)
'''
if anchor2 not in s:
    raise SystemExit("Could not find colors clamp line.")
s = s.replace(anchor2, insert2, 1)

# 3) Change interpolation factor from 1 -> 2
if "camtoworlds_all, 1" not in s:
    raise SystemExit("Could not find interpolation factor.")
s = s.replace("camtoworlds_all, 1", "camtoworlds_all, 2", 1)

p.write_text(s)
print("Created simple_trainer_modified.py from working simple_trainer.py")
PY

In [ ]:
# Extract frames (from better checkpoint)
%%bash
source /content/py310/bin/activate
cd /content/gsplat/examples

export MPLBACKEND=Agg
export PYTHONUNBUFFERED=1

# Run the modified version of simple_trainer.py; Choose between 6999 and 29999
python -u simple_trainer_modified.py default \
  --data_dir /content/nvs/colmap_no_mask \
  --data_factor 4 \
  --result_dir ./results/no_mask \
  --ckpt ./results/no_mask/ckpts/ckpt_6999_rank0.pt \
  --render_traj_path interp \
  --disable_viewer

In [ ]:
# !zip -r /content/step_6999_frames.zip \
#     /content/gsplat/examples/results/no_mask/traj_frames/step_6999

In [ ]:
# from google.colab import files
# files.download('/content/step_6999_frames.zip')